# Inspection des données du robot (LiDAR + Caméra + Commandes)

Ce notebook permet de visualiser frame par frame les données enregistrées dans `dataset_webots.hdf5`.
Il affiche :
1. Le scan LiDAR (vue de dessus)
2. L'image caméra (reconstruite à partir des secteurs HSV si disponible, ou raw)
3. Les commandes moteurs (Brutes et Normalisées)
4. Les capteurs de proximité

In [24]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pickle

FILENAME = 'dataset_webots.hdf5'
FEATURE_MODE = 'lidar_camera_prox'  # options: 'lidar', 'camera', 'prox', 'lidar_camera', 'lidar_camera_prox'

print(f"Chargement de '{FILENAME}'...")

def to_2d_features(data):
    arr = np.asarray(data, dtype=np.float32)
    return arr.reshape(arr.shape[0], -1)

def normalize_input_01(arr):
    arr = np.asarray(arr, dtype=np.float32)
    max_val = np.nanmax(arr)
    if max_val > 1.0:
        arr = arr / max_val
    return np.clip(arr, 0.0, 1.0)

with h5py.File(FILENAME, 'r') as f:
    available_keys = list(f.keys())
    print('Clés disponibles dans le dataset :', available_keys)

    thymio_commands = np.asarray(f['thymio_commands'][:], dtype=np.float32)

    feature_blocks = []

    if 'lidar' in FEATURE_MODE:
        lidar_key_candidates = ['thymio_scans', 'thymio_lidar', 'lidar']
        lidar_key = next((k for k in lidar_key_candidates if k in f), None)
        if lidar_key is None:
            raise KeyError(f"Aucune clé LiDAR trouvée parmi {lidar_key_candidates}")
        lidar_raw = f[lidar_key][:]
        lidar_2d = normalize_input_01(to_2d_features(lidar_raw))
        feature_blocks.append(lidar_2d)
        print(f"LiDAR utilisé: {lidar_key}, shape brut={np.asarray(lidar_raw).shape}, shape 2D={lidar_2d.shape}")

    if 'camera' in FEATURE_MODE:
        camera_key_candidates = [
            'thymio_cam',
            'thymio_camera_sectors_hsv',
            'thymio_camera_hsv',
            'thymio_camera',
            'camera',
            'camera_hsv'
        ]
        camera_key = next((k for k in camera_key_candidates if k in f), None)
        if camera_key is None:
            raise KeyError(f"Aucune clé caméra trouvée parmi {camera_key_candidates}")
        camera_raw = f[camera_key][:]
        camera_2d = normalize_input_01(to_2d_features(camera_raw))
        feature_blocks.append(camera_2d)
        print(f"Caméra utilisée: {camera_key}, shape brut={np.asarray(camera_raw).shape}, shape 2D={camera_2d.shape}")

    if 'prox' in FEATURE_MODE:
        prox_key_candidates = ['thymio_prox', 'prox']
        prox_key = next((k for k in prox_key_candidates if k in f), None)
        if prox_key is None:
            raise KeyError(f"Aucune clé prox trouvée parmi {prox_key_candidates}")
        prox_raw = f[prox_key][:]
        prox_2d = normalize_input_01(to_2d_features(prox_raw))
        feature_blocks.append(prox_2d)
        print(f"Prox utilisé: {prox_key}, shape brut={np.asarray(prox_raw).shape}, shape 2D={prox_2d.shape}")

    if not feature_blocks:
        raise ValueError("Aucune entrée sélectionnée. Utiliser FEATURE_MODE avec 'lidar', 'camera' et/ou 'prox'.")

    X = np.concatenate(feature_blocks, axis=1) if len(feature_blocks) > 1 else feature_blocks[0]
    y = np.clip(thymio_commands / 2.0, -1.0, 1.0)

print(f"Dataset chargé : {len(y)} échantillons")
print(f"Exemple commandes brutes : {thymio_commands[0]}")
print(f"Plage X : {X.min():.3f} / {X.max():.3f}")
print(f"Plage y (cmd norm.) : {y.min():.3f} / {y.max():.3f}")
print(f"Shape finale X : {X.shape}")

# 3. Partitionnement Train / Test (Scikit-learn)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"Taille Train : {X_train.shape[0]}")
print(f"Taille Test  : {X_test.shape[0]}")

Chargement de 'dataset_webots.hdf5'...
Clés disponibles dans le dataset : ['thymio_cam', 'thymio_commands', 'thymio_prox', 'thymio_scans']
LiDAR utilisé: thymio_scans, shape brut=(95463, 90, 2), shape 2D=(95463, 180)
Caméra utilisée: thymio_cam, shape brut=(95463, 9), shape 2D=(95463, 9)
Prox utilisé: thymio_prox, shape brut=(95463, 7), shape 2D=(95463, 7)
Dataset chargé : 95463 échantillons
Exemple commandes brutes : [0.2 0.2]
Plage X : 0.000 / 1.000
Plage y (cmd norm.) : -1.000 / 1.000
Shape finale X : (95463, 196)
Taille Train : 76370
Taille Test  : 19093


## 9c. Entraînement du MLP (Réseau à deux couches cachées)

L'énoncé demande un réseau avec **deux couches cachées**. Nous utilisons `MLPRegressor` (régression des vitesses moteurs).

Les entrées sont configurables dans la cellule 2 via `FEATURE_MODE` :
- `lidar` : LiDAR uniquement
- `camera` : caméra uniquement
- `prox` : capteurs de proximité uniquement
- `lidar_camera` : fusion LiDAR + caméra
- `lidar_camera_prox` : fusion complète (LiDAR + caméra + prox)

La préparation convertit automatiquement toutes les entrées en matrice 2D `(n_samples, n_features)` pour éviter les erreurs de dimension.

In [25]:
# 4. Configuration et Entraînement
mlp = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        hidden_layer_sizes=(64, 64),  # 2 couches cachées
        activation='relu',
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=2000,
        tol=1e-5,
        n_iter_no_change=30,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
        verbose=True
    ))
])

print("Début de l'entraînement...")
mlp.fit(X_train, y_train)

trained_mlp = mlp.named_steps['mlp']
print(f"Entraînement terminé en {trained_mlp.n_iter_} itérations.")
print(f"Loss finale : {trained_mlp.loss_:.6f}")

Début de l'entraînement...
Iteration 1, loss = 0.03132666
Validation score: 0.810438
Iteration 2, loss = 0.01243448
Validation score: 0.856642
Iteration 3, loss = 0.01010673
Validation score: 0.874932
Iteration 4, loss = 0.00886024
Validation score: 0.885413
Iteration 5, loss = 0.00813648
Validation score: 0.890515
Iteration 6, loss = 0.00758290
Validation score: 0.897126
Iteration 7, loss = 0.00707507
Validation score: 0.903384
Iteration 8, loss = 0.00682153
Validation score: 0.910575
Iteration 9, loss = 0.00640406
Validation score: 0.914338
Iteration 10, loss = 0.00621290
Validation score: 0.917138
Iteration 11, loss = 0.00602041
Validation score: 0.922031
Iteration 12, loss = 0.00586958
Validation score: 0.920245
Iteration 13, loss = 0.00570340
Validation score: 0.924423
Iteration 14, loss = 0.00558730
Validation score: 0.927115
Iteration 15, loss = 0.00544675
Validation score: 0.925801
Iteration 16, loss = 0.00534618
Validation score: 0.929514
Iteration 17, loss = 0.00523818
Valida

## 9d. Evaluation

À la convergence, on évalue le réseau avec `score` sur l'ensemble de TEST.
Objectif demandé : **$R^2 > 0.97$**.

In [26]:
# 5. Evaluation
score = mlp.score(X_test, y_test)
print(f"Score R^2 sur TEST : {score:.4f}")

# Petit aperçu des prédictions
sample_idx = 100
predicted = mlp.predict([X_test[sample_idx]])
truth = y_test[sample_idx]

print(f"Exemple n°{sample_idx} :")
print(f"  Vrai : {truth}")
print(f"  Prédiction : {predicted[0]}")
print(f"  Erreur : {np.abs(predicted[0] - truth)}")

Score R^2 sur TEST : 0.9654
Exemple n°100 :
  Vrai : [1.         0.71019596]
  Prédiction : [1.00733565 0.86454497]
  Erreur : [0.00733565 0.15434901]


## 9e. Sauvegarde du Modèle

Si le score est satisfaisant (> 0.97), on sauvegarde le modèle dans un fichier `.model` avec Pickle.
Nous allons sauvegarder le modèle qui prédit des valeurs normalisées ([-1, 1]).
Côté robot, il faudra penser à multiplier la sortie par **2.0** pour retrouver la vraie commande.

In [27]:
# 6. Sauvegarde
filename = 'ai_controller_model_hyper_prox.model'

# Consigne : sauvegarder seulement si score > 0.94
if score > 0.94:
    print(f"Score satisfaisant ({score:.4f} > 0.94). Sauvegarde en cours...")
    with open(filename, 'wb') as model_file:
        pickle.dump(mlp, model_file)
    print(f"Modèle sauvegardé dans : {filename}")
else:
    print(f"Score insuffisant ({score:.4f} <= 0.94). Modèle NON sauvegardé.")
    print("Ajustez la topologie, le nombre d'itérations ou les hyperparamètres.")

Score satisfaisant (0.9654 > 0.94). Sauvegarde en cours...
Modèle sauvegardé dans : ai_controller_model_hyper_prox.model


In [28]:
# Diagnostic rapide
from sklearn.metrics import mean_squared_error

train_score = mlp.score(X_train, y_train)
test_score = mlp.score(X_test, y_test)
y_pred_test = mlp.predict(X_test)

est = mlp.named_steps['mlp'] if hasattr(mlp, 'named_steps') else mlp
print(f"n_iter: {est.n_iter_}")
print(f"loss finale: {est.loss_:.6f}")
print(f"R2 train: {train_score:.4f}")
print(f"R2 test : {test_score:.4f}")
print("MSE test (moteur gauche, moteur droit):", mean_squared_error(y_test, y_pred_test, multioutput='raw_values'))

n_iter: 333
loss finale: 0.002583
R2 train: 0.9727
R2 test : 0.9654
MSE test (moteur gauche, moteur droit): [0.00613196 0.0053076 ]
